<a href="https://colab.research.google.com/github/zainabbio/zainabbio/blob/main/Internship.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **HLA-DR4BindPred: Predicting HLA-DRB1*04:01 Binding Peptides Using Machine Learning**

**Human Leukocyte Antigen (HLA) System**

The Human Leukocyte Antigen (HLA) system is a group of genes located on chromosome 6 that encode proteins responsible for presenting peptides to immune cells. These proteins are essential for immune recognition and play a critical role in distinguishing self from non-self.

HLA molecules are divided into two main classes:

**HLA Class I** (HLA-A, HLA-B, HLA-C): Present peptides to CD8+ T cells

**HLA Class II** (HLA-DR, HLA-DQ, HLA-DP): Present peptides to CD4+ T cells

Among these, HLA-DRB1*04:01 (HLA-DR4) is an important allele associated with several autoimmune diseases.

**What is HLA-DRB1*04:01?**

*   HLA-DRB1*04:01 is a specific allele of the HLA-DR gene that encodes a class II  MHC molecule.

*   It has been associated with multiple diseases, including: Rheumatoid Arthritis, Type 1 Diabetes, Multiple Sclerosis, COVID-19 immune responses
*   HLA-DR molecules bind peptides derived from pathogens or self proteins and present them to CD4+ T cells, which activate immune responses.

Therefore, identifying which peptides bind to HLA-DRB1*04:01 is important for understanding immune responses and developing therapies.

**What is a Peptide?**

A peptide is a short chain of amino acids.

For example:
AAAKLVFFAE

Peptides typically range between 8–20 amino acids.

In immunology:


*   Proteins from pathogens are broken into peptides.
*   These peptides bind to HLA molecules.

*   The peptide-HLA complex is recognized by T cells.
*   If the peptide binds strongly, it can trigger an immune response.



**Why Predict HLA Binding Peptides using Machine Learning?**

Identifying HLA-binding peptides experimentally is:


*   Expansive
*   Time-consuming
*   Labor intensive
*   List item


Machine learning can help predict binding peptides computationally.

Applications include:


1.   Vaccine design: Identify immunogenic peptides from pathogens.
2.   Immunotherapy: Develop peptide-based treatments.
3.   Autoimmune disease research: Identify peptides that trigger abnormal immune responses.
4.   Drug development: Understand peptide-HLA interactions.

**What is Machine Learning?**

Machine Learning (ML) is a field of artificial intelligence that enables computers to learn patterns from data and make predictions.

**Machine Learning Workflow**

1. Data collection
2. Feature extraction
3. Model training
4. Model evaluation
5. Prediction

**Feature Encoding**

Protein sequences must be converted into numerical features.
In HLA-DR4BindPred we use:

1. CPSR (Composite Protein Sequence Representation)
2. ExPseAAC (Extended Pseudo Amino Acid Composition)
3. DDE (Dipeptide Deviation from Expected Mean)

These capture: amino acid composition, sequence order, physicochemical properties

**Note** : In this notebook, we will focus only on Composite Protein Sequence Representation (CPSR) because it is simple, intuitive, and effective for representing peptide sequences.

**CPSR (Composite Protein Sequence Representation)**

Composite Protein Sequence Representation (CPSR) is a feature encoding method used in bioinformatics to convert protein or peptide sequences into numerical vectors.

CPSR combines multiple sequence descriptors to capture different aspects of peptide information: Amino acid composition, Sequence order information, Physicochemical properties

By combining these features, CPSR provides a comprehensive representation of peptide sequences.


**Machine Learning Models for Peptide Binding Prediction**

After converting peptide sequences into numerical features using Composite Protein Sequence Representation (CPSR), the next step is to train machine learning models to classify peptides as binders or non-binders to *HLA‑DRB104:01.

Machine learning models learn patterns from labeled data and then use these patterns to make predictions on unseen peptides.

In this notebook, we will explore the following models:

1. CatBoost
2. Random Forest
3. XGBoost
4. HGBoost
5. MLP
6. Support Vector Machine (SVM)
7. LDA

**1. CatBoost**

CatBoost is a gradient boosting algorithm developed by Yandex. It is designed to handle complex datasets efficiently and often provides strong predictive performance.

**Install Required Libraries**

Some libraries such as CatBoost and LightGBM may not be pre-installed in the Google Colab environment. Therefore, we install them first.

In [1]:
# Install required external libraries
!pip install catboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.1 MB/s eta 0:00:00


**Import Required Libraries**

Next, we import the necessary Python libraries for data handling, preprocessing, and machine learning.

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import os
import scipy.io as sio

# Sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import scale

# Boosting libraries
from catboost import CatBoostClassifier
import lightgbm as lgb

In [3]:
# Install required libraries (run once in Colab)
!pip install catboost lightgbm

# ===============================
# Imports
# ===============================
import pandas as pd
import numpy as np
import os
import math
import joblib

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import scale
from sklearn.metrics import confusion_matrix, roc_auc_score

from catboost import CatBoostClassifier

# ===============================
# Load Data
# ===============================
train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')
train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

# Stack positive and negative samples
train_esm1b = np.row_stack((train_esm1b_P, train_esm1b_N))

# Create labels properly
label = np.array([1]*len(train_esm1b_P) + [0]*len(train_esm1b_N))

# ===============================
# Standardization
# ===============================
X = scale(train_esm1b)
y = label

# ===============================
# Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# ===============================
# Train Model
# ===============================
model = CatBoostClassifier(verbose=0)
model.fit(X_train, y_train)

# Save model
joblib.dump(model, 'model.pkl')

# ===============================
# Cross Validation (5-Fold)
# ===============================
def confusion_matrix_scorer(clf, X_data, y_data):
    y_pred = clf.predict(X_data)
    cm = confusion_matrix(y_data, y_pred)
    return {
        'tn': cm[0, 0],
        'fp': cm[0, 1],
        'fn': cm[1, 0],
        'tp': cm[1, 1]
    }

cv_results = cross_validate(
    model, X_train, y_train, cv=5,
    scoring=confusion_matrix_scorer
)

TP = cv_results['test_tp'].mean()
FN = cv_results['test_fn'].mean()
FP = cv_results['test_fp'].mean()
TN = cv_results['test_tn'].mean()

# ===============================
# TRAINING METRICS
# ===============================
accuracy = (TP+TN) / (TP+TN+FP+FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)

# Safe MCC calculation
denominator = math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))
MCC = ((TP*TN) - (FP*FN)) / denominator if denominator != 0 else 0

print("\n===== TRAINING METRICS =====")
print("Accuracy:", accuracy)
print("F1_score:", F1_score)
print("Precision:", precision)
print("Specificity:", specificity)
print("Sensitivity/Recall:", sensitivity_recall)
print("MCC:", MCC)

# ===============================
# TESTING METRICS
# ===============================
pred_test = model.predict(X_test)
conf = confusion_matrix(y_test, pred_test)

TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

accuracy = (TP+TN) / (TP+TN+FP+FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)

denominator = math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))
MCC = ((TP*TN) - (FP*FN)) / denominator if denominator != 0 else 0

print("\n===== TESTING METRICS =====")
print("Accuracy:", accuracy)
print("F1_score:", F1_score)
print("Precision:", precision)
print("Specificity:", specificity)
print("Sensitivity/Recall:", sensitivity_recall)
print("MCC:", MCC)

# ===============================
# AUC (Testing)
# ===============================
y_test_pred_proba = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_pred_proba)

print("Testing AUC:", auc_test)

/tmp/ipykernel_352/1828646972.py:26: DeprecationWarning: `row_stack` alias is deprecated. Use `np.vstack` directly.
  train_esm1b = np.row_stack((train_esm1b_P, train_esm1b_N))


X_train: (16223, 71)
X_test: (4056, 71)
y_train: (16223,)
y_test: (4056,)

===== TRAINING METRICS =====
Accuracy: 0.7266843370523332
F1_score: 0.7248696947133284
Precision: 0.7256802087215802
Specificity: 0.7292790583619422
Sensitivity/Recall: 0.7240609892153217
MCC: 0.45334668303112774

===== TESTING METRICS =====
Accuracy: 0.7327416173570019
F1_score: 0.7312840852751611
Precision: 0.7323733862959285
Specificity: 0.7352652259332023
Sensitivity/Recall: 0.7301980198019802
MCC: 0.46547071556384423
Testing AUC: 0.810146326518703


**2. Random Forest**

Random Forest is an ensemble learning method that constructs multiple decision trees and combines their predictions.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score
import math
import joblib

# Load data
train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')
train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

# Stack positive and negative samples
train_esm1b = np.vstack((train_esm1b_P, train_esm1b_N))  # Fixed deprecation warning

# Labels
label1 = np.ones((train_esm1b_P.shape[0], 1))
label2 = np.zeros((train_esm1b_N.shape[0], 1))
label = np.append(label1, label2)

# Standardize the dataset
scaler = MinMaxScaler()
scaler.fit(train_esm1b)
shu = scale(train_esm1b)

X = shu
y = label

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Train RandomForest
model = RandomForestClassifier(random_state=42).fit(X_train, y_train)

# Save model
joblib.dump(model, 'RFmodel.pkl')

# Cross-validation confusion matrix scorer
def confusion_matrix_scorer(clf, X_train, y_train):
    y_pred = clf.predict(X_train)
    cm = confusion_matrix(y_train, y_pred)
    return {'tn': cm[0, 0], 'fp': cm[0, 1],
            'fn': cm[1, 0], 'tp': cm[1, 1]}

cv_results = cross_validate(model, X_train, y_train, cv=5,
                            scoring=confusion_matrix_scorer)

# Get mean TP, TN, FP, FN
TP = np.mean(cv_results['test_tp'])
FN = np.mean(cv_results['test_fn'])
FP = np.mean(cv_results['test_fp'])
TN = np.mean(cv_results['test_tn'])

# Metrics for training
accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Training Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)


#### TESTING ####
pred_test = model.predict(X_test)
conf = confusion_matrix(y_test, pred_test)
TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Testing Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

# AUC for Testing
y_test_pred_proba = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_pred_proba)
print("Testing AUC: ", auc_test)

(16223, 71)
(4056, 71)
(16223,)
(4056,)
Training Metrics:
Accuracy:  0.7565801639647414
F1_score:  0.750741652464811
Precision:  0.7647890946502058
Specificity:  0.7757479156449241
Sensitivity/Recall:  0.7372009421098302
MCC:  0.5133804555728775
Testing Metrics:
Accuracy:  0.7716962524654832
F1_score:  0.7659251769464105
Precision:  0.7825413223140496
Specificity:  0.7932220039292731
Sensitivity/Recall:  0.75
MCC:  0.543777605405224
Testing AUC:  0.8551953451730242


**3. XGBoost**

XGBoost is an ensemble machine learning method that builds multiple decision trees sequentially. Each new tree learns from the errors made by the previous trees, improving the overall model performance.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
import math
import joblib

# Load data
train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')
train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

# Stack positive and negative samples
train_esm1b = np.vstack((train_esm1b_P, train_esm1b_N))  # Fixed deprecation warning

# Labels
label1 = np.ones((train_esm1b_P.shape[0], 1))
label2 = np.zeros((train_esm1b_N.shape[0], 1))
label = np.append(label1, label2)

# Standardize the dataset
scaler = MinMaxScaler()
scaler.fit(train_esm1b)
shu = scale(train_esm1b)

X = shu
y = label

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Train XGBoost
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# Save model
joblib.dump(model, 'XGBoostmodel.pkl')

# Cross-validation confusion matrix scorer
def confusion_matrix_scorer(clf, X_train, y_train):
    y_pred = clf.predict(X_train)
    cm = confusion_matrix(y_train, y_pred)
    return {'tn': cm[0, 0], 'fp': cm[0, 1],
            'fn': cm[1, 0], 'tp': cm[1, 1]}

cv_results = cross_validate(model, X_train, y_train, cv=5,
                            scoring=confusion_matrix_scorer)

# Get mean TP, TN, FP, FN
TP = np.mean(cv_results['test_tp'])
FN = np.mean(cv_results['test_fn'])
FP = np.mean(cv_results['test_fp'])
TN = np.mean(cv_results['test_tn'])

# Training metrics
accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Training Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

#### TESTING ####
pred_test = model.predict(X_test)
conf = confusion_matrix(y_test, pred_test)
TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Testing Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

# Testing AUC
y_test_pred_proba = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_pred_proba)
print("Testing AUC: ", auc_test)

(16223, 71)
(4056, 71)
(16223,)
(4056,)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [19:57:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [19:57:35] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [19:57:36] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [19:57:37] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

Training Metrics:
Accuracy:  0.7227393207174999
F1_score:  0.7227905830149144
Precision:  0.7187155288638314
Specificity:  0.7186120647376165
Sensitivity/Recall:  0.7269121110697905
MCC:  0.44552511027114694
Testing Metrics:
Accuracy:  0.7374260355029586
F1_score:  0.7376201034737621
Precision:  0.7341834232466895
Specificity:  0.7337917485265226
Sensitivity/Recall:  0.7410891089108911
MCC:  0.47488414826322234
Testing AUC:  0.8125385389717754


**4. HGBoost**

HGBoost (Hierarchical Gradient Boosting) is a machine learning library that combines gradient boosting with hierarchical classification. It is designed for datasets where classes may have hierarchical relationships or where boosting models need better interpretability and structure.

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier as HGBoost
import math
import joblib

# Load data
train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')
train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

# Stack positive and negative samples
train_esm1b = np.vstack((train_esm1b_P, train_esm1b_N))  # Fixed deprecation warning

# Labels
label1 = np.ones((train_esm1b_P.shape[0], 1))
label2 = np.zeros((train_esm1b_N.shape[0], 1))
label = np.append(label1, label2)

# Standardize the dataset
scaler = MinMaxScaler()
scaler.fit(train_esm1b)
shu = scale(train_esm1b)

X = shu
y = label

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Train HistGradientBoostingClassifier
model = HGBoost(random_state=42)
model.fit(X_train, y_train)

# Save model
joblib.dump(model, 'HistGBoostmodel.pkl')

# Cross-validation confusion matrix scorer
def confusion_matrix_scorer(clf, X_train, y_train):
    y_pred = clf.predict(X_train)
    cm = confusion_matrix(y_train, y_pred)
    return {'tn': cm[0, 0], 'fp': cm[0, 1],
            'fn': cm[1, 0], 'tp': cm[1, 1]}

cv_results = cross_validate(model, X_train, y_train, cv=5,
                            scoring=confusion_matrix_scorer)

# Get mean TP, TN, FP, FN
TP = np.mean(cv_results['test_tp'])
FN = np.mean(cv_results['test_fn'])
FP = np.mean(cv_results['test_fp'])
TN = np.mean(cv_results['test_tn'])

# Training metrics
accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Training Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

#### TESTING ####
pred_test = model.predict(X_test)
conf = confusion_matrix(y_test, pred_test)
TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Testing Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

# Testing AUC
y_test_pred_proba = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_pred_proba)
print("Testing AUC: ", auc_test)

/usr/local/lib/python3.12/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:19: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


(16223, 71)
(4056, 71)
(16223,)
(4056,)
Training Metrics:
Accuracy:  0.7123836528385625
F1_score:  0.7101143141153082
Precision:  0.7117947440528086
Specificity:  0.7162824914173614
Sensitivity/Recall:  0.7084417999256228
MCC:  0.42473986888891907
Testing Metrics:
Accuracy:  0.7140039447731755
F1_score:  0.7134387351778656
Precision:  0.7120315581854043
Specificity:  0.7131630648330058
Sensitivity/Recall:  0.7148514851485148
MCC:  0.4280112197509802
Testing AUC:  0.7935597366219923


**5. MLP**

MLP (Multi-Layer Perceptron) is a type of artificial neural network (ANN) used for machine learning tasks such as classification and regression.

It is one of the simplest and most widely used neural network models.

An MLP consists of multiple layers of neurons that learn patterns from input data.


In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.neural_network import MLPClassifier
import math
import joblib

# Load data
train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')
train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

# Stack positive and negative samples
train_esm1b = np.vstack((train_esm1b_P, train_esm1b_N))  # Fixed deprecation warning

# Labels
label1 = np.ones((train_esm1b_P.shape[0], 1))
label2 = np.zeros((train_esm1b_N.shape[0], 1))
label = np.append(label1, label2)

# Standardize the dataset
scaler = MinMaxScaler()
scaler.fit(train_esm1b)
shu = scale(train_esm1b)

X = shu
y = label

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Train MLPClassifier
model = MLPClassifier(max_iter=500, random_state=42)
model.fit(X_train, y_train)

# Save model
joblib.dump(model, 'MLPmodel.pkl')

# Cross-validation confusion matrix scorer
def confusion_matrix_scorer(clf, X_train, y_train):
    y_pred = clf.predict(X_train)
    cm = confusion_matrix(y_train, y_pred)
    return {'tn': cm[0, 0], 'fp': cm[0, 1],
            'fn': cm[1, 0], 'tp': cm[1, 1]}

cv_results = cross_validate(model, X_train, y_train, cv=5,
                            scoring=confusion_matrix_scorer)

# Get mean TP, TN, FP, FN
TP = np.mean(cv_results['test_tp'])
FN = np.mean(cv_results['test_fn'])
FP = np.mean(cv_results['test_fp'])
TN = np.mean(cv_results['test_tn'])

# Training metrics
accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Training Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)


#### TESTING ####
pred_test = model.predict(X_test)
conf = confusion_matrix(y_test, pred_test)
TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Testing Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

# Testing AUC
y_test_pred_proba = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_pred_proba)
print("Testing AUC: ", auc_test)

(16223, 71)
(4056, 71)
(16223,)
(4056,)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Training Metrics:
Accuracy:  0.6810084448005918
F1_score:  0.6848931376727759
Precision:  0.6730493058879847
Specificity:  0.6650318783717509
Sensitivity/Recall:  0.6971612743275071
MCC:  0.3623523497666843
Testing Metrics:
Accuracy:  0.6945266272189349
F1_score:  0.6993448192186362
Precision:  0.6858638743455497
Specificity:  0.6758349705304518
Sensitivity/Recall:  0.7133663366336633
MCC:  0.3894506694390831
Testing AUC:  0.7525195004765703


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


**6. Support Vector Machine**

Support Vector Machine (SVM) is a supervised machine learning algorithm used for classification and regression problems.

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.svm import SVC
import math
import joblib

# Load data
train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')
train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

# Stack positive and negative samples
train_esm1b = np.vstack((train_esm1b_P, train_esm1b_N))

# Labels
label1 = np.ones((train_esm1b_P.shape[0], 1))
label2 = np.zeros((train_esm1b_N.shape[0], 1))
label = np.append(label1, label2)

# Standardize dataset
scaler = MinMaxScaler()
scaler.fit(train_esm1b)
shu = scale(train_esm1b)

X = shu
y = label

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Train SVM
model = SVC(kernel='rbf', C=1, probability=True, random_state=42)
model.fit(X_train, y_train)

# Save model
joblib.dump(model, 'SVMmodel.pkl')

# Cross-validation confusion matrix scorer
def confusion_matrix_scorer(clf, X_train, y_train):
    y_pred = clf.predict(X_train)
    cm = confusion_matrix(y_train, y_pred)
    return {'tn': cm[0, 0], 'fp': cm[0, 1],
            'fn': cm[1, 0], 'tp': cm[1, 1]}

cv_results = cross_validate(model, X_train, y_train, cv=5,
                            scoring=confusion_matrix_scorer)

TP = np.mean(cv_results['test_tp'])
FN = np.mean(cv_results['test_fn'])
FP = np.mean(cv_results['test_fp'])
TN = np.mean(cv_results['test_tn'])

accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Training Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

#### TESTING ####
pred_test = model.predict(X_test)
conf = confusion_matrix(y_test, pred_test)

TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Testing Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

# Testing AUC
y_test_pred_proba = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_pred_proba)
print("Testing AUC: ", auc_test)

(16223, 71)
(4056, 71)
(16223,)
(4056,)
Training Metrics:
Accuracy:  0.7273007458546509
F1_score:  0.7223198594024605
Precision:  0.7315956770502224
Specificity:  0.7411721432074546
Sensitivity/Recall:  0.7132763108962439
MCC:  0.454651596627395
Testing Metrics:
Accuracy:  0.7329881656804734
F1_score:  0.7287753568745304
Precision:  0.7374556512924481
Specificity:  0.7455795677799607
Sensitivity/Recall:  0.7202970297029703
MCC:  0.46604439469757314
Testing AUC:  0.8034218230270964


**7. LDA**

LDA (Linear Discriminant Analysis) is a supervised machine learning algorithm used for classification. It is especially useful when you have labeled data and want to separate classes based on their features.

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import scale, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import math
import joblib

# Load data
train_esm1b_P = pd.read_csv('HLA_main_pos_CPCR.csv')
train_esm1b_N = pd.read_csv('HLA_main_neg_CPCR.csv')

# Stack positive and negative samples
train_esm1b = np.vstack((train_esm1b_P, train_esm1b_N))

# Labels
label1 = np.ones((train_esm1b_P.shape[0], 1))
label2 = np.zeros((train_esm1b_N.shape[0], 1))
label = np.append(label1, label2)

# Standardize dataset
scaler = MinMaxScaler()
scaler.fit(train_esm1b)
shu = scale(train_esm1b)

X = shu
y = label

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

# Train LDA
model = LinearDiscriminantAnalysis()
model.fit(X_train, y_train)

# Save model
joblib.dump(model, 'LDAmodel.pkl')

# Cross-validation confusion matrix scorer
def confusion_matrix_scorer(clf, X_train, y_train):
    y_pred = clf.predict(X_train)
    cm = confusion_matrix(y_train, y_pred)
    return {'tn': cm[0, 0], 'fp': cm[0, 1],
            'fn': cm[1, 0], 'tp': cm[1, 1]}

cv_results = cross_validate(model, X_train, y_train, cv=5,
                            scoring=confusion_matrix_scorer)

TP = np.mean(cv_results['test_tp'])
FN = np.mean(cv_results['test_fn'])
FP = np.mean(cv_results['test_fp'])
TN = np.mean(cv_results['test_tn'])

accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Training Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

#### TESTING ####
pred_test = model.predict(X_test)
conf = confusion_matrix(y_test, pred_test)

TP = conf[1, 1]
FP = conf[0, 1]
TN = conf[0, 0]
FN = conf[1, 0]

accuracy = (TP + TN) / (TP + TN + FP + FN)
F1_score = 2*TP / ((2*TP) + (FP + FN))
precision = TP / (TP + FP)
specificity = TN / (FP + TN)
sensitivity_recall = TP / (TP + FN)
MCC = ((TP*TN) - (FP*FN)) / math.sqrt(((TP+FP)*(TP+FN))*((TN+FP)*(TN+FN)))

print("Testing Metrics:")
print("Accuracy: ", accuracy)
print("F1_score: ", F1_score)
print("Precision: ", precision)
print("Specificity: ", specificity)
print("Sensitivity/Recall: ", sensitivity_recall)
print("MCC: ", MCC)

# Testing AUC
y_test_pred_proba = model.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_pred_proba)
print("Testing AUC: ", auc_test)

(16223, 71)
(4056, 71)
(16223,)
(4056,)
Training Metrics:
Accuracy:  0.6688651913949331
F1_score:  0.6697405631378335
Precision:  0.6643493108915721
Specificity:  0.6625796959293772
Sensitivity/Recall:  0.6752200322300732
MCC:  0.3378142998692463
Testing Metrics:
Accuracy:  0.6629684418145957
F1_score:  0.6618847390551571
Precision:  0.6613939693524469
Specificity:  0.6635559921414538
Sensitivity/Recall:  0.6623762376237624
MCC:  0.3259306844051434
Testing AUC:  0.7158401252698945


This notebook provides a complete workflow to predict HLA-DRB1*04:01 binding peptides using CPSR features and multiple machine learning models.